# Haru Colab - MKV Muxing & Extract Tool

**Jalur utama (disarankan):** jalankan **1A Setup**, lalu **1B** (install CLI + web terminal otomatis). Ketik `haru-mux` / `haru-extract` / `haru-metadata` di terminal yang terbuka. Semua alur download - edit - mux/extract - upload + notif Telegram ada di terminal.

**Jalur alternatif:** cell form satu-per-satu di bawah (download, register track, edit, mux, mediainfo, upload). Boleh diskip kalau pakai web terminal.
> Butuh downloader YouTube / LRC / MangaDex? Buka `aio.ipynb` (satu repo, pola pakai sama).


---

### Persiapan (sebelum pakai)

Buka menu **Rahasia** (ikon kunci di sidebar kiri), lalu tambah secret berikut (aktifkan toggle akses notebook-nya):

| `GOFILE_API_TOKEN` | `fb` | Token filmbeehub proxy (download/upload Gofile) |
| `GDRIVE_CLIENT_ID` | *(dari Google Cloud Console)* | OAuth Client ID untuk Google Drive |
| `GDRIVE_CLIENT_SECRET` | *(dari Google Cloud Console)* | OAuth Client Secret untuk Google Drive |
| `GDRIVE_REFRESH_TOKEN` | *(dari OAuth flow)* | OAuth Refresh Token untuk Google Drive |
| `OWNER_ID` | *(Telegram chat ID)* | Untuk auto-post link terminal & hasil ke Telegram |
| `HARU_BOT_TOKEN` | *(token BotFather)* | Token bot Telegram khusus Haru (jangan pakai BOT_TOKEN lain) |

> **Google Drive:** Jika sudah punya `GDRIVE_CLIENT_ID`, `GDRIVE_CLIENT_SECRET`, dan `GDRIVE_REFRESH_TOKEN`, cell Google Drive akan otomatis pakai auth tersebut. Jika belum, cukup klik **Hubungkan** saat cell pertama dijalankan (menggunakan auth bawaan Colab).

## 1 — Setup (Wajib)
Cukup jalankan cell ini satu kali! Semua tools (`haru-mux`, `haru-mirror`, `haru-extract`, `haru-metadata`, `haru-download`, `haru-upload`, `auto-rename`, `yazi`, `mc`) langsung siap digunakan di Terminal bawaan Colab (pojok kiri bawah).

In [ ]:
#@title 1 — Setup (Install Semua Tools & CLI) { display-mode: "form" }
import subprocess, os, sys, time, json, base64, shutil
from pathlib import Path

print('📦 [1/4] Menginstall paket sistem (apt)...')
subprocess.run(['apt-get', 'update', '-qq'], capture_output=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'mkvtoolnix', 'mediainfo', 'tmux', 'jq', 'tree', 'wget', 'curl', 'mc', 'unzip'], capture_output=True)

print('🐍 [2/4] Menginstall library Python...')
subprocess.run(['pip', 'install', '-q', 'gdown', 'requests', 'huggingface_hub', 'yt-dlp', 'colorama', 'cloudscraper'], capture_output=True)

# Install Yazi (modern TUI file manager)
print('📁 [3/4] Menginstall Yazi File Manager...')
if not os.path.exists('/usr/local/bin/yazi'):
    try:
        yazi_url = 'https://github.com/sxyazi/yazi/releases/latest/download/yazi-x86_64-unknown-linux-musl.zip'
        subprocess.run(['curl', '-s', '-L', yazi_url, '-o', '/tmp/yazi.zip'], check=True)
        subprocess.run(['unzip', '-q', '-o', '/tmp/yazi.zip', '-d', '/tmp/yazi_extracted'], check=True)
        for p in Path('/tmp/yazi_extracted').rglob('yazi'):
            if p.is_file() and os.access(p, os.X_OK):
                shutil.copy2(p, '/usr/local/bin/yazi')
                break
        subprocess.run(['chmod', '+x', '/usr/local/bin/yazi'])
    except Exception as _e:
        print('  Gagal install Yazi otomatis:', _e)

try:
    if os.path.exists('/usr/local/bin/yazi') and not os.path.exists('/usr/bin/yazi'):
        os.symlink('/usr/local/bin/yazi', '/usr/bin/yazi')
except Exception:
    pass

# Create directories
UPLOAD_DIR = Path('/content/uploads')
UPLOAD_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path('/content/output')
OUTPUT_DIR.mkdir(exist_ok=True)
Path('/content/input').mkdir(exist_ok=True)

print('⚡ [4/4] Memasang CLI tools Haru...')
TOOLS = {
    'haru-mux': """...""",
    'haru-mirror': """...""",
    'haru-extract': """...""",
    'haru-metadata': """...""",
    'haru-download': """...""",
    'haru-upload': """IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwppbXBvcnQgc3VicHJvY2VzcyxzeXMsb3MscmUsZ2xvYixqc29uLHRpbWUKaW1wb3J0IHJlcXVlc3RzCmZyb20gcGF0aGxpYiBpbXBvcnQgUGF0aApWPXsnLm1rdicsJy5tcDQnLCcuYXZpJywnLm1vdicsJy53ZWJtJywnLmZsdicsJy53bXYnLCcudHMnLCcubTR2J30KQT17Jy5tcDMnLCcuYWFjJywnLmZsYWMnLCcud2F2JywnLm9nZycsJy5vcHVzJywnLm1rYScsJy5hYzMnLCcuZHRzJywnLmVhYzMnLCcubTRhJ30KUz17Jy5zcnQnLCcuYXNzJywnLnNzYScsJy5zdWInLCcuaWR4JywnLnN1cCcsJy52dHQnLCcucGdzJywnLnNjYycsJy5zYW1pJ30KTD17J2lkJzonSW5kb25lc2lhbicsJ2VuJzonRW5nbGlzaCcsJ2phJzonSmFwYW5lc2UnLCdrbyc6J0tvcmVhbicsJ3poJzonQ2hpbmVzZScsJ21zJzonTWFsYXknLCdhcic6J0FyYWJpYycsJ2RlJzonR2VybWFuJywnZnInOidGcmVuY2gnLCdlcyc6J1NwYW5pc2gnLCdwdCc6J1BvcnR1Z3Vlc2UnLCdydSc6J1J1c3NpYW4nLCdpdCc6J0l0YWxpYW4nLCd0aCc6J1RoYWknLCd2aSc6J1ZpZXRuYW1lc2UnLCdoaSc6J0hpbmRpJywndW5kJzonVW5kZXRlcm1pbmVkJ30KVVBMT0FEPVBhdGgoJy9jb250ZW50L3VwbG9hZHMnKQpPVVRQVVQ9UGF0aCgnL2NvbnRlbnQvb3V0cHV0JykKVVBMT0FELm1rZGlyKGV4aXN0X29rPVRydWUpCk9VVFBVVC5ta2RpcihleGlzdF9vaz1UcnVlKQpkZWYgY2koKToKIGltcG9ydCBzeXMKIHN5cy5zdGRvdXQud3JpdGUoJ1x4MWJbMkpceDFiW0gnKQogc3lzLnN0ZG91dC5mbHVzaCgpCmRlZiBvayh0KTpyZXR1cm4gJ1wwMzNbOTJtJyt0KydcMDMzWzBtJwpkZWYgZXIodCk6cmV0dXJuICdcMDMzWzkxbScrdCsnXDAzM1swbScKZGVmIGRpbSh0KTpyZXR1cm4gJ1wwMzNbOTBtJyt0KydcMDMzWzBtJwpkZWYgaGRyKHRpdGxlKTpwcmludCgnXG4nKyc9Jyo2Mik7cHJpbnQoJyAgJyt0aXRsZSk7cHJpbnQoJz0nKjYyKQpkZWYgbG9hZF9zZWNyZXRzKCk6CiB0cnk6CiAgaWYgb3MucGF0aC5leGlzdHMoJy9jb250ZW50Ly5oYXJ1X3NlY3JldHMuanNvbicpOgogICBkPWpzb24ubG9hZChvcGVuKCcvY29udGVudC8uaGFydV9zZWNyZXRzLmpzb24nKSkKICAgZm9yIGssdiBpbiBkLml0ZW1zKCk6CiAgICBpZiB2IGFuZCBub3Qgb3MuZW52aXJvbi5nZXQoayk6b3MuZW52aXJvbltrXT1zdHIodikKIGV4Y2VwdDpwYXNzCmRlZiBnZXRfc2VjcmV0KGspOgogdj1vcy5lbnZpcm9uLmdldChrLCcnKQogaWYgdjpyZXR1cm4gdi5zdHJpcCgpCiB0cnk6CiAgZnJvbSBnb29nbGUuY29sYWIgaW1wb3J0IHVzZXJkYXRhCiAgdD11c2VyZGF0YS5nZXQoaykKICBpZiB0OnJldHVybiBzdHIodCkuc3RyaXAoKQogZXhjZXB0OnBhc3MKIHJldHVybiAnJwpkZWYgZ2V0X2dvZmlsZV90b2tlbigpOgogcmV0dXJuIGdldF9zZWNyZXQoJ0dPRklMRV9BUElfVE9LRU4nKQpkZWYgZ29maWxlX3VwbG9hZF9maWxlcyh0YXJnZXRzLCBmb2xkZXJfbmFtZT1Ob25lKToKIGlmIG5vdCB0YXJnZXRzOnJldHVybiBGYWxzZSxbXQogdG9rZW49Z2V0X2dvZmlsZV90b2tlbigpCiBpZiBub3QgdG9rZW46cmV0dXJuIEZhbHNlLFtdCiB0cnk6CiAgcj1yZXF1ZXN0cy5nZXQoJ2h0dHBzOi8vYXBpLmdvZmlsZS5pby9hY2NvdW50cycsdGltZW91dD0xNSkKICBkPXIuanNvbigpCiAgaWYgZC5nZXQoJ3N0YXR1cycpIT0nb2snOnJldHVybiBGYWxzZSxbXQogIGFjY291bnRfdG9rZW49ZFsnZGF0YSddWyd0b2tlbiddCiBleGNlcHQ6cmV0dXJuIEZhbHNlLFtdCiBmb2xkZXJfaWQ9Tm9uZQogaWYgZm9sZGVyX25hbWUgYW5kIGxlbih0YXJnZXRzKT4xOgogIHRyeToKICAgcj1yZXF1ZXN0cy5wb3N0KCdodHRwczovL2FwaS5nb2ZpbGUuaW8vY29udGVudHMnLGhlYWRlcnM9eydBdXRob3JpemF0aW9uJzonQmVhcmVyICcrYWNjb3VudF90b2tlbiwnQ29udGVudC1UeXBlJzonYXBwbGljYXRpb24vanNvbid9LGpzb249eyd0eXBlJzonZm9sZGVyJywndGl0bGUnOmZvbGRlcl9uYW1lfSx0aW1lb3V0PTE1KQogICBkPXIuanNvbigpCiAgIGlmIGQuZ2V0KCdzdGF0dXMnKT09J29rJzpmb2xkZXJfaWQ9ZFsnZGF0YSddWydpZCddCiAgZXhjZXB0OnBhc3MKIHNydj0nc3RvcmUxJwogdHJ5OgogIHN2PXJlcXVlc3RzLmdldCgnaHR0cHM6Ly9hcGkuZ29maWxlLmlvL3NlcnZlcnMnLGhlYWRlcnM9eydBdXRob3JpemF0aW9uJzonQmVhcmVyICcrYWNjb3VudF90b2tlbn0sdGltZW91dD0xNSkuanNvbigpCiAgaWYgc3YuZ2V0KCdzdGF0dXMnKT09J29rJzpzcnY9c3ZbJ2RhdGEnXVsnc2VydmVycyddWzBdWyduYW1lJ10KIGV4Y2VwdDpwYXNzCiBsaW5rcz1bXQogZm9yIGYgaW4gdGFyZ2V0czoKICBwcmludCgnICBVcGxvYWQgJytmLm5hbWUrJyAoJytzdHIocm91bmQoZi5zdGF0KCkuc3Rfc2l6ZS8xMDI0LzEwMjQsMSkpKydNQikgdmlhICcrc3J2KycuLi4nKQogIGNtZD1bJ2N1cmwnLCctcycsJy1GJywnZmlsZT1AJytzdHIoZildCiAgaWYgZm9sZGVyX2lkOmNtZC5leHRlbmQoWyctRicsJ2ZvbGRlcklkPScrZm9sZGVyX2lkXSkKICBjbWQuYXBwZW5kKCdodHRwczovLycrc3J2KycuZ29maWxlLmlvL3VwbG9hZEZpbGUnKQogIHI9c3VicHJvY2Vzcy5ydW4oY21kLGNhcHR1cmVfb3V0cHV0PVRydWUsdGV4dD1UcnVlLHRpbWVvdXQ9NjAwKQogIHRyeToKICAgZGF0YT1qc29uLmxvYWRzKHIuc3Rkb3V0KQogICBpZiBkYXRhLmdldCgnc3RhdHVzJyk9PSdvayc6CiAgICBsaW5rcy5hcHBlbmQoKGYubmFtZSxkYXRhWydkYXRhJ11bJ2Rvd25sb2FkUGFnZSddKSkKICAgIHByaW50KCcgICcrb2soJ29rJykrJyAnK2YubmFtZSkKICAgZWxzZTpwcmludCgnICAnK2VyKCdnYWdhbCcpKycgJytzdHIoZGF0YSlbOjEwMF0pCiAgZXhjZXB0OnByaW50KCcgICcrZXIoJ2dhZ2FsJykrJyAnK2YubmFtZSsnIChubyByZXNwb25zZSknKQogaWYgbm90IGxpbmtzOnJldHVybiBGYWxzZSxbXQogaWYgbGVuKGxpbmtzKT09MTpyZXR1cm4gVHJ1ZSxbbGlua3NbMF1dCiByZXR1cm4gVHJ1ZSxsaW5rcwpkZWYgZ2RyaXZlX3NlY3JldChrKToKIHJldHVybiBnZXRfc2VjcmV0KGspCmRlZiBnZHJpdmVfdG9rZW4oY2lkLHNlYyxyZWYpOgogdHJ5OgogIHI9cmVxdWVzdHMucG9zdCgnaHR0cHM6Ly9vYXV0aDIuZ29vZ2xlYXBpcy5jb20vdG9rZW4nLGRhdGE9eydjbGllbnRfaWQnOmNpZCwnY2xpZW50X3NlY3JldCc6c2VjLCdyZWZyZXNoX3Rva2VuJzpyZWYsJ2dyYW50X3R5cGUnOidyZWZyZXNoX3Rva2VuJ30sdGltZW91dD0xNSkKICByZXR1cm4gci5qc29uKCkuZ2V0KCdhY2Nlc3NfdG9rZW4nKQogZXhjZXB0OnJldHVybiBOb25lCmRlZiBnZHJpdmVfdXBsb2FkX2ZpbGUodG9rLGZwYXRoLHBhcmVudCk6CiBzaXplPWZwYXRoLnN0YXQoKS5zdF9zaXplCiBtZXRhPXsnbmFtZSc6ZnBhdGgubmFtZSwncGFyZW50cyc6W3BhcmVudF19CiB0cnk6CiAgcj1yZXF1ZXN0cy5wb3N0KCdodHRwczovL3d3dy5nb29nbGVhcGlzLmNvbS91cGxvYWQvZHJpdmUvdjMvZmlsZXM/dXBsb2FkVHlwZT1yZXN1bWFibGUnLGhlYWRlcnM9eydBdXRob3JpemF0aW9uJzonQmVhcmVyICcrdG9rLCdDb250ZW50LVR5cGUnOidhcHBsaWNhdGlvbi9qc29uJywnWC1VcGxvYWQtQ29udGVudC1UeXBlJzonYXBwbGljYXRpb24vb2N0ZXQtc3RyZWFtJywnWC1VcGxvYWQtQ29udGVudC1MZW5ndGgnOnN0cihzaXplKX0sZGF0YT1qc29uLmR1bXBzKG1ldGEpLHRpbWVvdXQ9MzApCiAgdXJpPXIuaGVhZGVycy5nZXQoJ0xvY2F0aW9uJykKICBpZiBub3QgdXJpOnByaW50KCcgIEdhZ2FsIG11bGFpIHNlc2kgdXBsb2FkLicpO3JldHVybiBGYWxzZQogZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOnByaW50KCcgIEVycm9yIGluaXNpYXNpOiAnK3N0cihlKVs6MTUwXSk7cmV0dXJuIEZhbHNlCiBDSD02NCoxMDI0KjEwMjQgaWYgc2l6ZT4xMDAqMTAyNCoxMDI0IGVsc2UgMTYqMTAyNCoxMDI0CiB1cD0wO3QwPXRpbWUudGltZSgpCiB0cnk6CiAgZmg9b3BlbihmcGF0aCwncmInKQogIHdoaWxlIHVwPHNpemU6CiAgIGNoPWZoLnJlYWQoQ0gpCiAgIGlmIG5vdCBjaDpicmVhawogICBlbmQ9dXArbGVuKGNoKS0xCiAgIHJyPXJlcXVlc3RzLnB1dCh1cmksaGVhZGVycz17J0NvbnRlbnQtUmFuZ2UnOidieXRlcyAnK3N0cih1cCkrJy0nK3N0cihlbmQpKycvJytzdHIoc2l6ZSksJ0NvbnRlbnQtTGVuZ3RoJzpzdHIobGVuKGNoKSl9LGRhdGE9Y2gsdGltZW91dD0xMjApCiAgIGlmIHJyLnN0YXR1c19jb2RlIGluICgyMDAsMjAxKTp1cCs9bGVuKGNoKTticmVhawogICBlbGlmIHJyLnN0YXR1c19jb2RlPT0zMDg6CiAgICB1cCs9bGVuKGNoKQogICAgZWw9dGltZS50aW1lKCktdDA7c3A9dXAvZWwvMTAyNC8xMDI0IGlmIGVsPjAgZWxzZSAwCiAgICBwcmludCgnICAnK3N0cihyb3VuZCh1cC9zaXplKjEwMCwxKSkrJyUgICcrc3RyKHJvdW5kKHNwLDEpKSsnIE1CL3MnKQogICBlbHNlOnByaW50KCcgIFVwbG9hZCBlcnJvciBIVFRQICcrc3RyKHJyLnN0YXR1c19jb2RlKSk7ZmguY2xvc2UoKTtyZXR1cm4gRmFsc2UKICBmaC5jbG9zZSgpCiBleGNlcHQgRXhjZXB0aW9uIGFzIGU6cHJpbnQoJyAgRXJyb3IgdXBsb2FkOiAnK3N0cihlKVs6MTUwXSk7cmV0dXJuIEZhbHNlCiBwcmludChvaygnICAxMDAlIFNlbGVzYWkuJykpCiByZXR1cm4gVHJ1ZQpkZWYgdXBsb2FkX2dvZmlsZSgpOgogaGRyKCdVUExPQUQgLSBHb2ZpbGUnKQogYWxsX2ZpbGVzPVtdCiBmb3IgZCBpbiBbVVBMT0FELE9VVFBVVCxQYXRoKCcvY29udGVudC9leHRyYWN0cycpLFBhdGgoJy9jb250ZW50L2Rvd25sb2FkcycpXToKICBpZiBkLmV4aXN0cygpOgogICBmb3IgZiBpbiBzb3J0ZWQoZC5yZ2xvYignKicpKToKICAgIGlmIGYuaXNfZmlsZSgpIGFuZCBmLnN1ZmZpeC5sb3dlcigpIGluIFZ8QXxTOmFsbF9maWxlcy5hcHBlbmQoKGQsZikpCiBpZiBub3QgYWxsX2ZpbGVzOnByaW50KGVyKCcgIFRpZGFrIGFkYSBmaWxlIHVudHVrIGRpLXVwbG9hZC4nKSk7aW5wdXQoJyAgRW50ZXIuLi4nKTtyZXR1cm4KIHByaW50KCkKIGlkeD0wCiBmb3IgZCBpbiBbVVBMT0FELE9VVFBVVCxQYXRoKCcvY29udGVudC9leHRyYWN0cycpLFBhdGgoJy9jb250ZW50L2Rvd25sb2FkcycpXToKICBncnA9WyhkZCxmKSBmb3IgZGQsZiBpbiBhbGxfZmlsZXMgaWYgZGQ9PWRdCiAgaWYgbm90IGdycDpjb250aW51ZQogIHByaW50KCcgIFsnK2QubmFtZSsnL10gICgnK3N0cihsZW4oZ3JwKSkrJyBmaWxlKScpCiAgZm9yIGRkLGYgaW4gZ3JwOgogICBzaXplPWYuc3RhdCgpLnN0X3NpemUvMTAyNC8xMDI0CiAgIHByaW50KCcgICAgWycrc3RyKGlkeCkrJ10gJytmLm5hbWUrJyAgJytkaW0oc3RyKGludChzaXplKSkrJ01CJykpCiAgIGlkeCs9MQogIHByaW50KCkKIGZsYXQ9W2YgZm9yIGRkLGYgaW4gYWxsX2ZpbGVzXQogYz1pbnB1dCgnICBQaWxpaCAoKiBzZW11YSAvIDAsMSwyIC8gMC0zIC8gUSBiYXRhbCk6ICcpLnN0cmlwKCkudXBwZXIoKQogaWYgYz09J1EnOnJldHVybgogaWYgYz09JyonOnRhcmdldHM9ZmxhdAogZWxzZToKICB0cnk6CiAgIG51bXM9W10KICAgZm9yIHBhcnQgaW4gYy5zcGxpdCgnLCcpOgogICAgcGFydD1wYXJ0LnN0cmlwKCkKICAgIGlmICctJyBpbiBwYXJ0OmEsYj1wYXJ0LnNwbGl0KCctJywxKTtudW1zLmV4dGVuZChyYW5nZShpbnQoYSksaW50KGIpKzEpKQogICAgZWxzZTpudW1zLmFwcGVuZChpbnQocGFydCkpCiAgIHRhcmdldHM9W2ZsYXRbbl0gZm9yIG4gaW4gbnVtcyBpZiAwPD1uPGxlbihmbGF0KV0KICBleGNlcHQ6cHJpbnQoJyAgSW5wdXQgdGlkYWsgdmFsaWQuJyk7aW5wdXQoJyAgRW50ZXIuLi4nKTtyZXR1cm4KICBpZiBub3QgdGFyZ2V0czpyZXR1cm4KIGlmIGxlbih0YXJnZXRzKT4xOgogIGZuYW1lPWlucHV0KCcgIE5hbWEgZm9sZGVyIFsnK3RhcmdldHNbMF0ucGFyZW50Lm5hbWUrJ106ICcpLnN0cmlwKCkgb3IgdGFyZ2V0c1swXS5wYXJlbnQubmFtZQogZWxzZTpmbmFtZT1Ob25lCiBvayxsaW5rcz1nb2ZpbGVfdXBsb2FkX2ZpbGVzKHRhcmdldHMsZm5hbWUpCiBpZiBsaW5rczoKICBtc2c9JzxiPlVwbG9hZCBHb2ZpbGU8L2I+JwogIGZvciBuYW1lLHVybCBpbiBsaW5rczoKICAgcHJpbnQob2soJyAgJytuYW1lKSkKICAgcHJpbnQoJyAgJyt1cmwrJ1xuJykKICAgbXNnPW1zZysnXG4nK25hbWUrJ1xuJyt1cmwKICB0Z19zZW5kKG1zZykKIGVsc2U6cHJpbnQoZXIoJyAgU2VtdWEgdXBsb2FkIGdhZ2FsLicpKQogaW5wdXQoJ1xuICBFbnRlci4uLicpCmRlZiB1cGxvYWRfZHJpdmUoKToKIGhkcignVVBMT0FEIC0gR29vZ2xlIERyaXZlJykKIGFsbF9maWxlcz1bXQogZm9yIGQgaW4gW1VQTE9BRCxPVVRQVVQsUGF0aCgnL2NvbnRlbnQvZXh0cmFjdHMnKSxQYXRoKCcvY29udGVudC9kb3dubG9hZHMnKV06CiAgaWYgZC5leGlzdHMoKToKICAgZm9yIGYgaW4gc29ydGVkKGQucmdsb2IoJyonKSk6CiAgICBpZiBmLmlzX2ZpbGUoKSBhbmQgZi5zdWZmaXgubG93ZXIoKSBpbiBWfEF8UzphbGxfZmlsZXMuYXBwZW5kKChkLGYpKQogaWYgbm90IGFsbF9maWxlczpwcmludChlcignICBUaWRhayBhZGEgZmlsZSB1bnR1ayBkaS11cGxvYWQuJykpO2lucHV0KCcgIEVudGVyLi4uJyk7cmV0dXJuCiBwcmludCgpCiBpZHg9MAogZm9yIGQgaW4gW1VQTE9BRCxPVVRQVVQsUGF0aCgnL2NvbnRlbnQvZXh0cmFjdHMnKSxQYXRoKCcvY29udGVudC9kb3dubG9hZHMnKV06CiAgZ3JwPVsoZGQsZikgZm9yIGRkLGYgaW4gYWxsX2ZpbGVzIGlmIGRkPT1kXQogIGlmIG5vdCBncnA6Y29udGludWUKICBwcmludCgnICBbJytkLm5hbWUrJy9dICAoJytzdHIobGVuKGdycCkpKycgZmlsZSknKQogIGZvciBkZCxmIGluIGdycDoKICAgc2l6ZT1mLnN0YXQoKS5zdF9zaXplLzEwMjQvMTAyNAogICBwcmludCgnICAgIFsnK3N0cihpZHgpKyddICcrZi5uYW1lKycgICcrZGltKHN0cihpbnQoc2l6ZSkpKydNQicpKQogICBpZHgrPTEKICBwcmludCgpCiBmbGF0PVtmIGZvciBkZCxmIGluIGFsbF9maWxlc10KIGM9aW5wdXQoJyAgUGlsaWggKCogc2VtdWEgLyAwLDEsMiAvIDAtMyAvIFEgYmF0YWwpOiAnKS5zdHJpcCgpLnVwcGVyKCkKIGlmIGM9PSdRJzpyZXR1cm4KIGlmIGM9PScqJzp0YXJnZXRzPWZsYXQKIGVsc2U6CiAgdHJ5OgogICBudW1zPVtdCiAgIGZvciBwYXJ0IGluIGMuc3BsaXQoJywnKToKICAgIHBhcnQ9cGFydC5zdHJpcCgpCiAgICBpZiAnLScgaW4gcGFydDphLGI9cGFydC5zcGxpdCgnLScsMSk7bnVtcy5leHRlbmQocmFuZ2UoaW50KGEpLGludChiKSsxKSkKICAgIGVsc2U6bnVtcy5hcHBlbmQoaW50KHBhcnQpKQogICB0YXJnZXRzPVtmbGF0W25dIGZvciBuIGluIG51bXMgaWYgMDw9bjxsZW4oZmxhdCldCiAgZXhjZXB0OnByaW50KCcgIElucHV0IHRpZGFrIHZhbGlkLicpO2lucHV0KCcgIEVudGVyLi4uJyk7cmV0dXJuCiAgaWYgbm90IHRhcmdldHM6cmV0dXJuCiBjaWQ9Z2RyaXZlX3NlY3JldCgnR0RSSVZFX0NMSUVOVF9JRCcpO3NlYz1nZHJpdmVfc2VjcmV0KCdHRFJJVkVfQ0xJRU5UX1NFQ1JFVCcpO3JlZj1nZHJpdmVfc2VjcmV0KCdHRFJJVkVfUkVGUkVTSF9UT0tFTicpCiBwYXJlbnRfaWQ9Z2RyaXZlX3NlY3JldCgnR0RSSVZFX0ZPTERFUl9JRCcpIG9yICcxcGpwZDYzUFRGdndZZDhpSTdkdk13Y1UtZV9MTXF2VUUnCiBpZiBub3QoY2lkIGFuZCBzZWMgYW5kIHJlZik6CiAgcHJpbnQoZXIoJyAgU2VjcmV0IEdEcml2ZSB0aWRhayBrZWJhY2EuJykpO3ByaW50KCcgIEFrdGlma2FuIHRvZ2dsZSBzZWNyZXQgKyByZS1ydW4gY2VsbCBJbnN0YWxsLicpO2lucHV0KCcgIEVudGVyLi4uJyk7cmV0dXJuCiBwcmludCgnICBBdXRoIHZpYSBBUEkuLi4nKQogdG9rPWdkcml2ZV90b2tlbihjaWQsc2VjLHJlZikKIGlmIG5vdCB0b2s6cHJpbnQoZXIoJyAgR2FnYWwgZGFwYXQgYWNjZXNzIHRva2VuLicpKTtyZXR1cm4KIG09cmUuc2VhcmNoKHInL2ZvbGRlcnMvKFtBLVphLXowLTlfLV0rKScscGFyZW50X2lkKQogaWYgbTpwYXJlbnRfaWQ9bS5ncm91cCgxKQogZWxpZiBsZW4ocGFyZW50X2lkKTwyMDoKICBxPSJuYW1lPSciK3BhcmVudF9pZCsiJyBhbmQgbWltZVR5cGU9J2FwcGxpY2F0aW9uL3ZuZC5nb29nbGUtYXBwcy5mb2xkZXInIGFuZCB0cmFzaGVkPWZhbHNlIgogIHRyeToKICAgcj1yZXF1ZXN0cy5nZXQoJ2h0dHBzOi8vd3d3Lmdvb2dsZWFwaXMuY29tL2RyaXZlL3YzL2ZpbGVzJyxoZWFkZXJzPXsnQXV0aG9yaXphdGlvbic6J0JlYXJlciAnK3Rva30scGFyYW1zPXsncSc6cSwnZmllbGRzJzonZmlsZXMoaWQpJ30sdGltZW91dD0xNSkKICAgZnM9ci5qc29uKCkuZ2V0KCdmaWxlcycsW10pCiAgIGlmIGZzOnBhcmVudF9pZD1mc1swXVsnaWQnXQogIGV4Y2VwdDpwYXNzCiBzdWI9aW5wdXQoJyAgU3ViZm9sZGVyIFsnK2RpbSgnbGFuZ3N1bmcga2UgcGFyZW50JykrJ106ICcpLnN0cmlwKCkKIHRhcmdldD1wYXJlbnRfaWQKIGlmIHN1YjoKICB0cnk6CiAgIHEyPSJuYW1lPSciK3N1YisiJyBhbmQgJyIrcGFyZW50X2lkKyInIGluIHBhcmVudHMgYW5kIG1pbWVUeXBlPSdhcHBsaWNhdGlvbi92bmQuZ29vZ2xlLWFwcHMuZm9sZGVyJyBhbmQgdHJhc2hlZD1mYWxzZSIKICAgcjI9cmVxdWVzdHMuZ2V0KCdodHRwczovL3d3dy5nb29nbGVhcGlzLmNvbS9kcml2ZS92My9maWxlcycsaGVhZGVycz17J0F1dGhvcml6YXRpb24nOidCZWFyZXIgJyt0b2t9LHBhcmFtcz17J3EnOnEyLCdmaWVsZHMnOidmaWxlcyhpZCknfSx0aW1lb3V0PTE1KQogICBmczI9cjIuanNvbigpLmdldCgnZmlsZXMnLFtdKQogICBpZiBmczI6dGFyZ2V0PWZzMlswXVsnaWQnXQogICBlbHNlOgogICAgbWV0YT17J25hbWUnOnN1YiwnbWltZVR5cGUnOidhcHBsaWNhdGlvbi92bmQuZ29vZ2xlLWFwcHMuZm9sZGVyJywncGFyZW50cyc6W3BhcmVudF9pZF19CiAgICByMz1yZXF1ZXN0cy5wb3N0KCdodHRwczovL3d3dy5nb29nbGVhcGlzLmNvbS9kcml2ZS92My9maWxlcycsaGVhZGVycz17J0F1dGhvcml6YXRpb24nOidCZWFyZXIgJyt0b2ssJ0NvbnRlbnQtVHlwZSc6J2FwcGxpY2F0aW9uL2pzb24nfSxkYXRhPWpzb24uZHVtcHMobWV0YSksdGltZW91dD0xNSkKICAgIG5pZD1yMy5qc29uKCkuZ2V0KCdpZCcpCiAgICBpZiBuaWQ6dGFyZ2V0PW5pZDtwcmludCgnICBTdWJmb2xkZXIgZGlidWF0OiAnK3N1YikKICAgIGVsc2U6cHJpbnQoZXIoJyAgR2FnYWwgYnVhdCBzdWJmb2xkZXIuJykpCiAgZXhjZXB0OnByaW50KGVyKCcgIEVycm9yIGJ1YXQgc3ViZm9sZGVyLicpKQogb2tfbj0wO2ZhaWw9W10KIGZvciBmIGluIHRhcmdldHM6CiAgcHJpbnQoJyAgVXBsb2FkICcrZi5uYW1lKycgKCcrc3RyKHJvdW5kKGYuc3RhdCgpLnN0X3NpemUvMTAyNC8xMDI0LDEpKSsnTUIpLi4uJykKICBpZiBnZHJpdmVfdXBsb2FkX2ZpbGUodG9rLGYsdGFyZ2V0KTpva19uKz0xO3ByaW50KCcgICcrb2soJ29rJykrJyAnK2YubmFtZSkKICBlbHNlOmZhaWwuYXBwZW5kKGYubmFtZSk7cHJpbnQoJyAgJytlcignZ2FnYWwnKSsnICcrZi5uYW1lKQogaWYgb2tfbjp0Z19zZW5kKCc8Yj5VcGxvYWQgR0RyaXZlPC9iPlxuJytzdHIob2tfbikrJyBmaWxlIGJlcmhhc2lsJykKIGlmIGZhaWw6cHJpbnQoZXIoJyAgR2FnYWw6ICcrJywgJy5qb2luKGZhaWwpKSkKIGlucHV0KCdcbiAgRW50ZXIuLi4nKQpkZWYgdGdfb3duZXIoKToKIHJldHVybiBnZXRfc2VjcmV0KCdPV05FUl9JRCcpCmRlZiB0Z190b2tlbigpOgogcmV0dXJuIGdldF9zZWNyZXQoJ0hBUlVfQk9UX1RPS0VOJykKZGVmIHRnX3NlbmQobXNnKToKIG9pZD10Z19vd25lcigpO3Rvaz10Z190b2tlbigpCiBpZiBub3Qgb2lkIG9yIG5vdCB0b2s6cmV0dXJuCiB0cnk6cmVxdWVzdHMucG9zdCgnaHR0cHM6Ly9hcGkudGVsZWdyYW0ub3JnL2JvdCcrdG9rKycvc2VuZE1lc3NhZ2UnLGpzb249eydjaGF0X2lkJzpvaWQsJ3RleHQnOm1zZywncGFyc2VfbW9kZSc6J0hUTUwnLCdkaXNhYmxlX3dlYl9wYWdlX3ByZXZpZXcnOlRydWV9LHRpbWVvdXQ9MTApCiBleGNlcHQ6cGFzcwpkZWYgbWVudV91cGxvYWQoKToKIHdoaWxlIFRydWU6CiAgY2koKTtoZHIoJ1VQTE9BRCcpCiAgcHJpbnQoKQogIHByaW50KCcgIFsxXSBHb2ZpbGUgIChmb2xkZXIgZ2FidW5nYW4pJykKICBwcmludCgnICBbMl0gR29vZ2xlIERyaXZlIChtdWx0aS1maWxlICsgc3ViZm9sZGVyKScpCiAgcHJpbnQoKQogIHByaW50KCcgIFswXSBLZW1iYWxpJykKICBwcmludCgpCiAgYz1pbnB1dCgnICBQaWxpaDogJykuc3RyaXAoKQogIGlmIGM9PScwJzpyZXR1cm4KICBlbGlmIGM9PScxJzp1cGxvYWRfZ29maWxlKCkKICBlbGlmIGM9PScyJzp1cGxvYWRfZHJpdmUoKQogIGlucHV0KCdcbiAgRW50ZXIuLi4nKQpkZWYgbWFpbigpOgogbG9hZF9zZWNyZXRzKCkKIG1lbnVfdXBsb2FkKCkKaWYgX19uYW1lX189PSdfX21haW5fXyc6bWFpbigpCg==""",
    'auto-rename': """IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwppbXBvcnQgc3VicHJvY2VzcyxzeXMsb3MscmUsZ2xvYixqc29uLHRpbWUKZnJvbSBwYXRobGliIGltcG9ydCBQYXRoClY9eycubWt2JywnLm1wNCcsJy5hdmknLCcubW92JywnLndlYm0nLCcuZmx2JywnLndtdicsJy50cycsJy5tNHYnfQpBPXsnLm1wMycsJy5hYWMnLCcuZmxhYycsJy53YXYnLCcub2dnJywnLm9wdXMnLCcubWthJywnLmFjMycsJy5kdHMnLCcuZWFjMycsJy5tNGEnfQpTPXsnLnNydCcsJy5hc3MnLCcuc3NhJywnLnN1YicsJy5pZHgnLCcuc3VwJywnLnZ0dCcsJy5wZ3MnLCcuc2NjJywnLnNhbWknfQpBTExfRVhUPVZ8QXxTClVQTE9BRD1QYXRoKCcvY29udGVudC91cGxvYWRzJykKT1VUUFVUPVBhdGgoJy9jb250ZW50L291dHB1dCcpCkVYVFJBQ1RTPVBhdGgoJy9jb250ZW50L2V4dHJhY3RzJykKZGVmIGNpKCk6CiBpbXBvcnQgc3lzCiBzeXMuc3Rkb3V0LndyaXRlKCdceDFiWzJKXHgxYltIJykKIHN5cy5zdGRvdXQuZmx1c2goKQpkZWYgb2sodCk6cmV0dXJuICdcMDMzWzkybScrdCsnXDAzM1swbScKZGVmIGVyKHQpOnJldHVybiAnXDAzM1s5MW0nK3QrJ1wwMzNbMG0nCmRlZiBkaW0odCk6cmV0dXJuICdcMDMzWzkwbScrdCsnXDAzM1swbScKZGVmIGhkcih0aXRsZSk6cHJpbnQoJ1xuJysnPScqNjIpO3ByaW50KCcgICcrdGl0bGUpO3ByaW50KCc9Jyo2MikKCmRlZiBjbGVhbl9maWxlbmFtZShuYW1lKToKIG5hbWU9bmFtZS5zdHJpcCgpCiBuYW1lPXJlLnN1YihyJ1xbKFtBLVphLXowLTldKylcXScscidbXDFdICcsbmFtZSkKIG5hbWU9cmUuc3ViKHInXHMrJywnICcsbmFtZSkKIG5hbWU9cmUuc3ViKHInXChEdWFsIEF1ZGlvXCknLCcoRHVhbC1BdWRpbyknLG5hbWUpCiBuYW1lPXJlLnN1YihyJ1woRHVhbCBBdWRpbyAnLCcoRHVhbC1BdWRpbyAnLG5hbWUpCiBuYW1lPXJlLnN1YihyJyAoRHVhbCBBdWRpbykgJywnIChEdWFsLUF1ZGlvKSAnLG5hbWUpCiBtPXJlLnNlYXJjaChyJyg/PCFcZCkoXGR7MSwzfSkoPyFcZCknLG5hbWUpCiBpZiBtOgogIGVwPW0uZ3JvdXAoMSkuemZpbGwoMikKICBiZWZvcmU9bmFtZVs6bS5zdGFydCgpXQogIGFmdGVyPW5hbWVbbS5lbmQoKTpdCiAgaWYgbm90IHJlLnNlYXJjaChyJ1tTc11cZCtbRWVdXGQrJyxuYW1lKToKICAgc2Vhc29uPScwMScKICAgc209cmUuc2VhcmNoKHInW1NzXShcZHsxLDJ9KScsYmVmb3JlKQogICBpZiBzbTpzZWFzb249c20uZ3JvdXAoMSkuemZpbGwoMikKICAgbmFtZT1iZWZvcmUrJ1MnK3NlYXNvbisnRScrZXArYWZ0ZXIKIG5hbWU9cmUuc3ViKHInXHMqXChccyonLCcgKCcsbmFtZSkKIG5hbWU9cmUuc3ViKHInXHMqXClccyonLCcpICcsbmFtZSkKIG5hbWU9cmUuc3ViKHInICArJywnICcsbmFtZSkKIG5hbWU9bmFtZS5zdHJpcCgpCiByZXR1cm4gbmFtZQoKZGVmIHBpY2tfZm9sZGVyKCk6CiBjaSgpO2hkcignQVVUTyBSRU5BTUUgLSBQaWxpaCBGb2xkZXInKQogcHJpbnQoKQogcHJpbnQoJyAgWzFdIC9jb250ZW50L3VwbG9hZHMnKQogcHJpbnQoJyAgWzJdIC9jb250ZW50L291dHB1dCcpCiBwcmludCgnICBbM10gL2NvbnRlbnQvZXh0cmFjdHMnKQogcHJpbnQoJyAgWzRdIFNlbXVhIGZvbGRlcicpCiBwcmludCgnICBbUV0gS2VtYmFsaScpCiBwcmludCgpCiBjPWlucHV0KCcgIFBpbGloOiAnKS5zdHJpcCgpLnVwcGVyKCkKIGlmIGM9PSdRJzpyZXR1cm4gTm9uZQogaWYgYz09JzEnOnJldHVybiBVUExPQUQKIGlmIGM9PScyJzpyZXR1cm4gT1VUUFVUCiBpZiBjPT0nMyc6cmV0dXJuIEVYVFJBQ1RTCiBpZiBjPT0nNCc6cmV0dXJuIFtVUExPQUQsT1VUUFVULEVYVFJBQ1RTXQogcmV0dXJuIE5vbmUKCmRlZiBzY2FuX2ZpbGVzKGZvbGRlcnMpOgogaWYgbm90IGlzaW5zdGFuY2UoZm9sZGVycyxsaXN0KTpmb2xkZXJzPVtmb2xkZXJzXQogZmlsZXM9W10KIGZvciBkIGluIGZvbGRlcnM6CiAgaWYgbm90IGQuZXhpc3RzKCk6Y29udGludWUKICBmb3IgcCBpbiBzb3J0ZWQoZC5yZ2xvYignKicpKToKICAgaWYgbm90IHAuaXNfZmlsZSgpOmNvbnRpbnVlCiAgIGlmIHAuc3VmZml4Lmxvd2VyKCkgaW4gQUxMX0VYVDoKICAgIGNsZWFuZWQ9Y2xlYW5fZmlsZW5hbWUocC5uYW1lKQogICAgaWYgY2xlYW5lZCE9cC5uYW1lOmZpbGVzLmFwcGVuZCgocCxjbGVhbmVkKSkKIHJldHVybiBmaWxlcwoKZGVmIHNob3dfZmlsZXMoZmlsZXMpOgogcHJpbnQoKQogcHJpbnQoJyAgTm8gIE9yaWdpbmFsICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgLT4gQ2xlYW5lZCcpCiBwcmludCgnICAnKyctJyo4MCkKIGZvciBpLChvcmlnLGNsZWFuZWQpIGluIGVudW1lcmF0ZShmaWxlcyk6CiAgcHJpbnQoJyAgJytzdHIoaSkubGp1c3QoNCkrb3JpZy5uYW1lWzo1MF0ubGp1c3QoNTIpKyctPiAnK2NsZWFuZWRbOjQwXSkKCmRlZiBkb19yZW5hbWUoZmlsZXMsc2VsPU5vbmUpOgogb2tfbj0wCiB0YXJnZXRzPWZpbGVzIGlmIHNlbCBpcyBOb25lIGVsc2UgWyhmaWxlc1tpXSkgZm9yIGkgaW4gc2VsIGlmIDA8PWk8bGVuKGZpbGVzKV0KIGZvciBvcmlnLGNsZWFuZWQgaW4gdGFyZ2V0czoKICBuZXdfcGF0aD1vcmlnLnBhcmVudC9jbGVhbmVkCiAgaWYgbmV3X3BhdGguZXhpc3RzKCkgYW5kIG5ld19wYXRoIT1vcmlnOgogICBwcmludCgnICBTa2lwIChleGlzdHMpOiAnK2NsZWFuZWQpO2NvbnRpbnVlCiAgdHJ5OgogICBvcmlnLnJlbmFtZShuZXdfcGF0aCkKICAgcHJpbnQoJyAgJytvaygnT0snKSsnICcrb3JpZy5uYW1lKycgLT4gJytjbGVhbmVkKQogICBva19uKz0xCiAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOnByaW50KCcgICcrZXIoJ0VSUicpKycgJytzdHIoZSlbOjYwXSkKIHByaW50KCdcbiAgUmVuYW1lZDogJytzdHIob2tfbikrJy8nK3N0cihsZW4odGFyZ2V0cykpKQoKZGVmIG1haW4oKToKIHdoaWxlIFRydWU6CiAgZm9sZGVycz1waWNrX2ZvbGRlcigpCiAgaWYgZm9sZGVycyBpcyBOb25lOnJldHVybgogIGZpbGVzPXNjYW5fZmlsZXMoZm9sZGVycykKICBpZiBub3QgZmlsZXM6CiAgIHByaW50KCcgIFRpZGFrIGFkYSBmaWxlIHlhbmcgcGVybHUgZGktcmVuYW1lLicpO2lucHV0KCcgIEVudGVyLi4uJyk7Y29udGludWUKICB3aGlsZSBUcnVlOgogICBjaSgpO2hkcignQVVUTyBSRU5BTUUnKQogICBzaG93X2ZpbGVzKGZpbGVzKQogICBwcmludCgpCiAgIHByaW50KCcgIFtZXSBSZW5hbWUgc2VtdWEgICBbbm9tb3JdIHBpbGloICgwLDIsNSkgICBbUF0gUHJldmlldyAgIFtGXSBHYW50aSBmb2xkZXIgICBbUV0gS2VtYmFsaScpCiAgIHByaW50KCkKICAgYz1pbnB1dCgnICA+ICcpLnN0cmlwKCkudXBwZXIoKQogICBpZiBjPT0nUSc6YnJlYWsKICAgaWYgYz09J0YnOmJyZWFrCiAgIGlmIGM9PSdQJzoKICAgIGZvciBvcmlnLGNsZWFuZWQgaW4gZmlsZXM6CiAgICAgcHJpbnQoJyAgJytvcmlnLm5hbWUpCiAgICAgcHJpbnQoJyAgICAtPiAnK2NsZWFuZWQpCiAgICAgcHJpbnQoKQogICAgaW5wdXQoJyAgRW50ZXIuLi4nKTtjb250aW51ZQogICBpZiBjPT0nWSc6CiAgICBkb19yZW5hbWUoZmlsZXMpCiAgICBpbnB1dCgnICBFbnRlci4uLicpO2NvbnRpbnVlCiAgIHRyeToKICAgIHNlbD1zZXQoKQogICAgZm9yIHBhcnQgaW4gYy5zcGxpdCgnLCcpOgogICAgIHBhcnQ9cGFydC5zdHJpcCgpCiAgICAgaWYgcGFydC5pc2RpZ2l0KCk6c2VsLmFkZChpbnQocGFydCkpCiAgICBkb19yZW5hbWUoZmlsZXMsc2VsKQogICAgaW5wdXQoJyAgRW50ZXIuLi4nKQogICBleGNlcHQ6cGFzcwoKaWYgX19uYW1lX189PSdfX21haW5fXyc6bWFpbigpCg=="""
}

for _name, _blob in TOOLS.items():
    _p = '/usr/local/bin/' + _name
    _code = base64.b64decode(_blob).decode('utf-8').replace('\r\n', '\n').replace('\r', '\n')
    if not _code.startswith('#!'):
        _code = '#!/usr/bin/env python3\n' + _code
    with open(_p, 'w', encoding='utf-8') as _f:
        _f.write(_code)
    os.chmod(_p, 0o755)
    # Symlink / copy to /usr/bin to guarantee PATH lookup everywhere
    try:
        _p_usr = '/usr/bin/' + _name
        if os.path.exists(_p_usr) or os.path.islink(_p_usr):
            try: os.remove(_p_usr)
            except Exception: pass
        os.symlink(_p, _p_usr)
    except Exception:
        try:
            shutil.copy2(_p, '/usr/bin/' + _name)
            os.chmod('/usr/bin/' + _name, 0o755)
        except Exception: pass
    print('  ✓ ' + _name)

# Configure aliases and PATH for ALL shells (interactive & login)
_all_tool_names = list(TOOLS.keys()) + ['yazi', 'mc']
_bashrc_entries = [
    "\n# Haru CLI PATH and Aliases",
    "export PATH=/usr/local/bin:/usr/bin:$PATH"
]
for _tn in _all_tool_names:
    _bashrc_entries.append(f"alias {_tn}='/usr/local/bin/{_tn}'")
_bashrc_entries.append("hash -r 2>/dev/null\n")
_bashrc_text = "\n".join(_bashrc_entries)

try:
    with open('/etc/bash.bashrc', 'a', encoding='utf-8') as _f:
        _f.write(_bashrc_text)
except Exception: pass

try:
    with open('/etc/profile.d/haru.sh', 'w', encoding='utf-8') as _f:
        _f.write(_bashrc_text)
    os.chmod('/etc/profile.d/haru.sh', 0o755)
except Exception: pass

for _rc in ['/root/.bashrc', os.path.expanduser('~/.bashrc'), '/root/.profile']:
    try:
        with open(_rc, 'a', encoding='utf-8') as _f:
            _f.write(_bashrc_text)
    except Exception: pass

# Read Colab Secrets and export to /content/.haru_secrets.json
try:
    _secrets = {}
    try:
        from google.colab import userdata as _ud
        for _k in ['GOFILE_API_TOKEN', 'GDRIVE_CLIENT_ID', 'GDRIVE_CLIENT_SECRET', 'GDRIVE_REFRESH_TOKEN', 'GDRIVE_FOLDER_ID', 'OWNER_ID', 'HARU_BOT_TOKEN', 'HF_TOKEN', 'HF_REPO_ID']:
            try:
                _v = _ud.get(_k)
                if _v: _secrets[_k] = str(_v).strip()
            except Exception: pass
    except Exception: pass
    for _k, _v in _secrets.items(): os.environ[_k] = _v
    if _secrets:
        _old = {}
        if os.path.exists('/content/.haru_secrets.json'):
            try: _old = json.load(open('/content/.haru_secrets.json'))
            except Exception: pass
        _old.update(_secrets)
        with open('/content/.haru_secrets.json', 'w', encoding='utf-8') as _sf:
            json.dump(_old, _sf)
        os.chmod('/content/.haru_secrets.json', 0o600)
        print('  🔐 Secrets tersinkronisasi: ' + ', '.join(sorted(_old.keys())))
    else:
        print('  ℹ️  (Belum ada Secret Colab yang aktif - dapat diatur di ikon kunci sebelah kiri)')
except Exception: pass

print()
print('=' * 66)
print('✅ Setup Selesai! Semua tools siap digunakan.')
print('📌 Cara pakai di Terminal bawaan Colab (pojok kiri bawah):')
print('   Klik tab "Terminal" di panel bawah, lalu ketik command:')
print('   • haru-mux        : Muxing MKV interaktif / batch')
print('   • haru-mirror     : Mirror GDrive / GoFile / Direct -> GDrive / HF')
print('   • haru-extract    : Ekstrak subtitle, audio, attachments')
print('   • haru-metadata   : Edit track & metadata MKV')
print('   • haru-download   : Download dari Gofile / GDrive / Direct URL')
print('   • haru-upload     : Upload ke Gofile / GDrive / HuggingFace')
print('   • yazi / mc       : File manager TUI modern & interaktif')
print('   • auto-rename     : Rename file batch otomatis')
print('=' * 66)
print('💡 Info: Jika ingin Web Terminal di tab browser terpisah,')
print('   silakan jalankan Cell "1B — Web Terminal".')

## 1B — Web Terminal (Opsional - Tab Browser Terpisah)
Jalankan cell ini jika ingin membuka terminal di tab browser baru melalui Cloudflare Tunnel. Jika cukup menggunakan terminal bawaan Colab (pojok kiri bawah), Anda **tidak perlu** menjalankan cell ini.

In [ ]:
#@title 1B — Buka Web Terminal (Opsional - Tab Browser Baru) { display-mode: "form" }
import subprocess, os, time, re, requests
from IPython.display import HTML, display

# Pastikan tools sudah terpasang
if not os.path.exists('/usr/local/bin/haru-mux'):
    print('⚠️ Tools belum terpasang. Harap jalankan Cell 1 (Setup) terlebih dahulu!')

# Refresh secrets
try:
    _s2 = {}
    try:
        from google.colab import userdata as _ud
        for _k in ['GOFILE_API_TOKEN', 'GDRIVE_CLIENT_ID', 'GDRIVE_CLIENT_SECRET', 'GDRIVE_REFRESH_TOKEN', 'GDRIVE_FOLDER_ID', 'OWNER_ID', 'HARU_BOT_TOKEN', 'HF_TOKEN', 'HF_REPO_ID']:
            try:
                _v = _ud.get(_k)
                if _v: _s2[_k] = str(_v).strip()
            except Exception: pass
    except Exception: pass
    if _s2:
        import json as _js
        try: _old = _js.load(open('/content/.haru_secrets.json'))
        except Exception: _old = {}
        _old.update(_s2)
        with open('/content/.haru_secrets.json', 'w') as _sf: _js.dump(_old, _sf)
        os.chmod('/content/.haru_secrets.json', 0o600)
except Exception: pass

print('🌐 Menyiapkan Web Terminal...')
if not os.path.exists('/usr/local/bin/cloudflared'):
    print('  Download cloudflared...')
    subprocess.run(['curl', '-s', '-L', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', '-o', '/usr/local/bin/cloudflared'])
    subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'])

if not os.path.exists('/usr/local/bin/ttyd'):
    print('  Download ttyd...')
    subprocess.run(['curl', '-s', '-L', 'https://github.com/tsl0922/ttyd/releases/latest/download/ttyd.x86_64', '-o', '/usr/local/bin/ttyd'])
    subprocess.run(['chmod', '+x', '/usr/local/bin/ttyd'])

subprocess.run(['pkill', '-f', 'ttyd'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared tunnel'], capture_output=True)
time.sleep(1)

# Configure tmux
subprocess.run(['tmux', 'set', '-g', 'history-limit', '50000'], capture_output=True)
subprocess.run(['tmux', 'set', '-g', 'mouse', 'on'], capture_output=True)
subprocess.run(['tmux', 'set', '-g', 'default-terminal', 'xterm-256color'], capture_output=True)

subprocess.Popen(['/usr/local/bin/ttyd', '-p', '7681', '-W', '-t', 'fontSize=15', 'tmux', 'new-session', '-A', '-s', 'haru', 'bash'], cwd='/content', stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

print('  Buka tunnel Cloudflare...')
cf = subprocess.Popen(['/usr/local/bin/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:7681'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
web_url = None
end = time.time() + 35
while time.time() < end:
    line = cf.stdout.readline()
    if not line:
        time.sleep(0.3)
        continue
    m = re.findall(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        web_url = m[-1]
        break

print()
print('=' * 62)
if web_url:
    print('WEB TERMINAL SIAP:')
    print('  ' + web_url)
    print()
    print('  Perintah: haru-mux | haru-mirror | haru-extract | yazi | mc')
    try:
        from google.colab import userdata as _ud
        _oid = _ud.get('OWNER_ID') or ''
    except Exception:
        _oid = os.environ.get('OWNER_ID') or ''
    try:
        from google.colab import userdata as _ud2
        _tg = _ud2.get('HARU_BOT_TOKEN') or ''
    except Exception:
        _tg = os.environ.get('HARU_BOT_TOKEN') or ''
    if _oid and _tg:
        try:
            requests.post('https://api.telegram.org/bot' + _tg + '/sendMessage', json={'chat_id': _oid, 'text': '<b>HaruColab terminal siap!</b>\nWeb: ' + web_url + '\nKetik: haru-mux / haru-mirror / haru-extract / yazi / mc', 'parse_mode': 'HTML', 'disable_web_page_preview': True}, timeout=8)
            print('  Notif Telegram terkirim.')
        except Exception as _e:
            print('  Gagal kirim Telegram: ' + str(_e)[:100])
    else:
        print('  (Aktifkan HARU_BOT_TOKEN & OWNER_ID di Secrets biar link auto-post ke Telegram.)')
    display(HTML('<a href="' + web_url + '" target="_blank" style="background:#238636;color:#fff;padding:12px 24px;text-decoration:none;border-radius:6px;font-weight:bold;display:inline-block;">Buka Web Terminal</a>'))
else:
    print('Gagal dapat URL tunnel. Jalankan ulang cell ini.')
print('=' * 62)
print()
print('Biarkan cell ini running agar tunnel tetap hidup.')
try:
    while True:
        time.sleep(30)
except KeyboardInterrupt:
    print('Web terminal ditutup.')

## 1C — Terminal di dalam Cell (colab-xterm)
Enak di HP & bisa fullscreen. Jalankan cell di bawah, terminal muncul di dalam cell — ketik `haru-mux` di sana.

In [ ]:
#@title Buka Terminal di Cell { display-mode: "form" }
!pip install colab-xterm -q
%load_ext colabxterm
%xterm


---
## Jalur alternatif — form per cell
Bagian bawah ini versi form satu-per-satu (alternatif web terminal di atas). Boleh diskip kalau sudah pakai `haru-mux` / `haru-extract`.

In [ ]:
#@title Gofile Downloader { display-mode: "form" }
#@markdown ### Pilih mode download
mode = "Folder (auto-detect semua file)" #@param ["Satu file", "Folder (auto-detect semua file)"]

#@markdown ---
#@markdown ### Isi link Gofile
gofile_url = "" #@param {type:"string"}
gofile_password = "" #@param {type:"string"}

#@markdown ---
#@markdown ### (Opsional) Nama file override — kosongkan untuk auto
gofile_filename = "" #@param {type:"string"}

import os, re, json, requests, subprocess, hashlib, urllib.parse
from pathlib import Path

GOFILE_PROXY_API = 'https://go.filmbeehub.workers.dev/api/v1/generate'
GOFILE_PROXY_DATA = 'https://go.filmbeehub.workers.dev/api/data'
UPLOAD_DIR = Path('/content/uploads')
UPLOAD_DIR.mkdir(exist_ok=True)

def get_gofile_token():
    try:
        from google.colab import userdata
        t = userdata.get('GOFILE_API_TOKEN')
        if t: return str(t).strip()
    except Exception: pass
    if os.path.exists('/content/.haru_secrets.json'):
        try:
            d = json.load(open('/content/.haru_secrets.json'))
            if d.get('GOFILE_API_TOKEN'): return str(d['GOFILE_API_TOKEN']).strip()
        except Exception: pass
    return os.environ.get('GOFILE_API_TOKEN') or 'fb'

def gofile_generate_link(url, token, password=''):
    # Generate direct download link via filmbeehub proxy
    if not token or token == 'None': token = 'fb'
    payload = json.dumps({
        'url': url,
        'password': password or '',
        'expiresInSeconds': 3600,
        'filePage': 0,
        'filePageSize': 100
    })
    endpoints = [
        'https://go.filmbeehub.workers.dev/api/v1/generate',
        'https://go.eithon.qzz.io/api/v1/generate'
    ]
    for ep in endpoints:
        try:
            cmd = [
                'curl', '-s', '-L', '--location-trusted',
                '-X', 'POST', ep,
                '-H', f'Authorization: Bearer {token}',
                '-H', 'Content-Type: application/json',
                '-H', 'User-Agent: Mozilla/5.0',
                '-d', payload
            ]
            p = subprocess.run(cmd, capture_output=True, text=True, timeout=35)
            m = re.search(r'(\{[\s\S]*\})', p.stdout.strip())
            if m:
                d = json.loads(m.group(1))
                if d.get('ok') or 'data' in d: return d
        except Exception: pass
        try:
            headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json', 'User-Agent': 'Mozilla/5.0'}
            r = requests.post(ep, data=payload, headers=headers, allow_redirects=True, timeout=35)
            d = r.json()
            if d.get('ok') or 'data' in d: return d
        except Exception: pass
    return {}

def gofile_get_folder_files(url, token, password=''):
    if not token: token = 'fb'
    result = gofile_generate_link(url, token, password)
    if not result.get('ok') and 'data' not in result:
        print(f'  ❌ Gagal generate proxy: {result.get("error", "unknown")}')
        return []

    data = result.get('data', {})
    if data.get('downloadLinks'):
        return data['downloadLinks']

    share_url = data.get('shareUrl', '')
    if share_url:
        share_id = share_url.rstrip('/').split('/')[-1]
        print(f'  🔗 Share ID: {share_id}')
        for base in ['https://go.filmbeehub.workers.dev/api/data', 'https://go.eithon.qzz.io/api/data']:
            try:
                cmd = ['curl', '-s', '-L', f'{base}/{share_id}', '-H', 'User-Agent: Mozilla/5.0']
                p = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
                m = re.search(r'(\{[\s\S]*\})', p.stdout.strip())
                if m:
                    folder_data = json.loads(m.group(1))
                    all_files = []
                    for group in folder_data.get('groups', []): all_files.extend(group.get('files', []))
                    if all_files: return all_files
            except Exception: pass
            try:
                resp = requests.get(f'{base}/{share_id}', headers={'User-Agent': 'Mozilla/5.0'}, timeout=30)
                folder_data = resp.json()
                all_files = []
                for group in folder_data.get('groups', []): all_files.extend(group.get('files', []))
                if all_files: return all_files
            except Exception: pass
    return []

def detect_type(filepath):
    ext = filepath.suffix.lower()
    video_exts = {'.mkv','.mp4','.avi','.mov','.webm','.flv','.wmv','.ts','.m4v'}
    audio_exts = {'.mp3','.aac','.flac','.wav','.ogg','.opus','.mka','.ac3','.dts','.eac3','.m4a'}
    sub_exts   = {'.srt','.ass','.ssa','.sub','.idx','.sup','.vtt','.pgs','.scc','.sami'}
    if ext in video_exts: return 'video'
    if ext in audio_exts: return 'audio'
    if ext in sub_exts:   return 'subtitle'
    return 'other'

def gofile_download_file(url, password='', token=None, fname_override=None):
    token = token or get_gofile_token() or 'fb'
    print('  🔗 Request direct link via FilmBee proxy...')
    result = gofile_generate_link(url, token, password)
    data = result.get('data', {})
    links = data.get('downloadLinks', [])
    if not links:
        print(f'  ❌ Gagal dapat link: {result.get("error", "tidak ada download link")}')
        return None
    link = links[0]
    direct_url = link['downloadUrl']
    fname = fname_override or link.get('name', '')
    print(f'  📥 Downloading {fname}...')
    resp = requests.get(direct_url, stream=True, timeout=600)
    resp.raise_for_status()
    if not fname:
        cd = resp.headers.get('Content-Disposition', '')
        m = re.search(r'filename[*]?=["\']?([^"\';\n]+)', cd)
        fname = urllib.parse.unquote(m.group(1).strip()) if m else hashlib.md5(url.encode()).hexdigest()[:12]
    dest = UPLOAD_DIR / fname
    total = 0
    with open(dest, 'wb') as f:
        for chunk in resp.iter_content(chunk_size=1024*1024):
            f.write(chunk)
            total += len(chunk)
    print(f'  ✅ {fname}  ({total:,} bytes / {total/1024/1024:.1f} MB)')
    return dest

def gofile_download_folder(url, password='', token=None):
    token = token or get_gofile_token() or 'fb'
    print('  🔍 Ambil daftar file di folder via FilmBee proxy...')
    folder_files = gofile_get_folder_files(url, token, password)
    if not folder_files:
        print('  ❌ Folder kosong atau tidak bisa diakses.')
        return []
    print(f'  📋 Ditemukan {len(folder_files)} file:')
    for f in folder_files:
        ft = detect_type(Path(f['name']))
        size_str = f.get('size', '?')
        print(f'     [{ft:<9}] {f["name"]}  ({size_str})')
    print()
    downloaded = []
    for i, f in enumerate(folder_files, 1):
        print(f'  [{i}/{len(folder_files)}] {f["name"]}')
        try:
            dl_url = f.get('downloadUrl', '')
            if not dl_url:
                print(f'    ⚠️  Tidak ada download URL, skip.')
                continue
            resp = requests.get(dl_url, stream=True, timeout=600)
            resp.raise_for_status()
            dest = UPLOAD_DIR / f['name']
            total = 0
            with open(dest, 'wb') as fh:
                for chunk in resp.iter_content(chunk_size=1024*1024):
                    fh.write(chunk)
                    total += len(chunk)
            ft = detect_type(dest)
            print(f'    ✅ [{ft:<9}] {f["name"]}  ({total:,} bytes)')
            downloaded.append(dest)
        except Exception as e:
            print(f'    ❌ Gagal: {e}')
        print()
    return downloaded

# ─── Jalankan ───
if gofile_url.strip():
    if mode.startswith('Folder'):
        gofile_download_folder(gofile_url.strip(), gofile_password)
    else:
        gofile_download_file(gofile_url.strip(), gofile_password, fname_override=gofile_filename or None)
else:
    print('⏭️  Isi gofile_url di form sebelah kanan, lalu jalankan ulang.')

## 3 — Download dari Google Drive

In [ ]:
#@title Google Drive Downloader { display-mode: "form" }
#@markdown ### Link Google Drive / Folder ID / Local Path
#@markdown Contoh URL: `https://drive.google.com/drive/folders/...` atau path lokal: `/content/drive/MyDrive/Movies/film.mkv`
gdrive_path = "" #@param {type:"string"}

#@markdown ---
#@markdown ### (Opsional) Nama file override — kosongkan untuk auto
gdrive_filename = "" #@param {type:"string"}

from pathlib import Path
import shutil, os, re, requests, subprocess, time

UPLOAD_DIR = Path('/content/uploads')
UPLOAD_DIR.mkdir(exist_ok=True)

def mount_drive():
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        print('✅ Google Drive mounted.')
    except Exception as e:
        print(f'❌ Gagal mount: {e}')

def get_secret(k):
    try:
        from google.colab import userdata
        v = userdata.get(k)
        if v: return str(v).strip()
    except Exception: pass
    return os.environ.get(k, '').strip()

def gdrive_token(cid, sec, ref):
    try:
        r = requests.post('https://oauth2.googleapis.com/token',
                          data={'client_id': cid, 'client_secret': sec,
                                'refresh_token': ref, 'grant_type': 'refresh_token'},
                          timeout=15)
        return r.json().get('access_token')
    except Exception:
        return None

def extract_gdrive_id(s):
    s = s.strip()
    m = re.search(r'/folders/([a-zA-Z0-9_-]+)', s)
    if m: return m.group(1), True
    m = re.search(r'/file/d/([a-zA-Z0-9_-]+)', s)
    if m: return m.group(1), False
    m = re.search(r'[?&]id=([a-zA-Z0-9_-]+)', s)
    if m: return m.group(1), None
    m = re.search(r'id=([a-zA-Z0-9_-]+)', s)
    if m: return m.group(1), None
    if re.match(r'^[a-zA-Z0-9_-]{20,}$', s): return s, None
    return None, None

def gdrive_download_file(tok, fid, name, size, dest_dir):
    dest = dest_dir / name
    part = dest_dir / (name + '.part')
    if dest.exists() and dest.stat().st_size > 0:
        if size and dest.stat().st_size == int(size):
            print(f'  SKIP {name} (sudah ada)')
            return True
    url = f'https://www.googleapis.com/drive/v3/files/{fid}?alt=media'
    headers = {'Authorization': f'Bearer {tok}'}
    try:
        r = requests.get(url, headers=headers, stream=True, timeout=30)
        if r.status_code != 200:
            print(f'  ❌ Gagal download {name}: HTTP {r.status_code}')
            return False
        total = int(size) if size else int(r.headers.get('content-length', 0))
        done = 0
        t0 = time.time()
        with open(part, 'wb') as fh:
            for ch in r.iter_content(chunk_size=16*1024*1024):
                if ch:
                    fh.write(ch)
                    done += len(ch)
                    el = time.time() - t0
                    spd = (done / el / 1024 / 1024) if el > 0 else 0
                    if total > 0:
                        pct = round(done / total * 100, 1)
                        print(f'\r    {pct}%  {round(done/1024/1024, 1)}MB  ({round(spd, 1)} MB/s)', end='', flush=True)
                    else:
                        print(f'\r    {round(done/1024/1024, 1)}MB  ({round(spd, 1)} MB/s)', end='', flush=True)
        print()
        if part.exists():
            if dest.exists(): dest.unlink()
            part.rename(dest)
            print(f'  ✅ {name} ({round(done/1024/1024, 1)} MB)')
            return True
    except Exception as e:
        print(f'\n  ❌ Error {name}: {e}')
        if part.exists():
            try: part.unlink()
            except: pass
    return False

def gdrive_list_folder(tok, folder_id):
    files = []
    page_token = None
    while True:
        params = {'q': f"'{folder_id}' in parents and trashed=false", 'fields': 'nextPageToken, files(id, name, mimeType, size)', 'pageSize': 1000}
        if page_token: params['pageToken'] = page_token
        try:
            r = requests.get('https://www.googleapis.com/drive/v3/files', headers={'Authorization': f'Bearer {tok}'}, params=params, timeout=20)
            d = r.json()
            if 'error' in d: return None
            files.extend(d.get('files', []))
            page_token = d.get('nextPageToken')
            if not page_token: break
        except Exception: return None
    return files

target_input = gdrive_path.strip()
if target_input:
    gid, is_folder_hint = extract_gdrive_id(target_input)
    # Jika berupa URL atau ID Google Drive
    if gid or target_input.startswith('http'):
        cid = get_secret('GDRIVE_CLIENT_ID')
        sec = get_secret('GDRIVE_CLIENT_SECRET')
        ref = get_secret('GDRIVE_REFRESH_TOKEN')
        tok = gdrive_token(cid, sec, ref) if (cid and sec and ref) else None
        if tok and gid:
            print('🔑 Menggunakan Google Drive OAuth API v3...')
            r = requests.get(f'https://www.googleapis.com/drive/v3/files/{gid}?fields=id,name,mimeType,size', headers={'Authorization': f'Bearer {tok}'}, timeout=15)
            item = r.json()
            if 'error' not in item:
                mime = item.get('mimeType', '')
                if mime == 'application/vnd.google-apps.folder' or is_folder_hint:
                    print(f'📂 Folder: {item.get("name", "drive_folder")}')
                    flist = gdrive_list_folder(tok, gid)
                    if flist:
                        flist = [f for f in flist if f.get('mimeType') != 'application/vnd.google-apps.folder']
                        print(f'📋 Ditemukan {len(flist)} file:')
                        for f in flist:
                            dest_n = gdrive_filename.strip() if gdrive_filename.strip() and len(flist)==1 else f['name']
                            gdrive_download_file(tok, f['id'], dest_n, f.get('size'), UPLOAD_DIR)
                    else:
                        print('❌ Folder kosong atau gagal mengambil daftar file.')
                else:
                    dest_n = gdrive_filename.strip() if gdrive_filename.strip() else item.get('name', 'file')
                    print(f'📥 Downloading file: {dest_n}...')
                    gdrive_download_file(tok, gid, dest_n, item.get('size'), UPLOAD_DIR)
            else:
                print('⚠️ ID tidak ditemukan di API, fallback ke gdown...')
                cmd = ['gdown', '-O', str(UPLOAD_DIR), '--remaining-ok', target_input]
                if is_folder_hint: cmd.insert(1, '--folder')
                subprocess.run(cmd)
        else:
            print('⚠️ OAuth secret tidak lengkap, mencoba download via gdown...')
            cmd = ['gdown', '-O', str(UPLOAD_DIR), '--remaining-ok', target_input]
            if is_folder_hint or '/folders/' in target_input: cmd.insert(1, '--folder')
            subprocess.run(cmd)
    else:
        # Local path
        src = Path(target_input)
        if not src.exists():
            mount_drive()
        if src.exists():
            if src.is_dir():
                print(f'📂 Copy semua file dari folder: {src}\n')
                for f in src.iterdir():
                    if f.is_file():
                        dest_name = gdrive_filename.strip() if gdrive_filename.strip() else f.name
                        shutil.copy2(f, UPLOAD_DIR / dest_name)
                        print(f'  ✅ {f.name}  →  {dest_name}')
            else:
                dest_name = gdrive_filename.strip() if gdrive_filename.strip() else src.name
                shutil.copy2(src, UPLOAD_DIR / dest_name)
                print(f'✅ {src.name}  →  {dest_name}')
        else:
            print(f'❌ Tidak ditemukan: {target_input}')
else:
    print('⏭️  Isi gdrive_path di form sebelah kanan, lalu jalankan ulang.')


## 4 — Download dari Direct URL

In [ ]:
#@title Direct URL Downloader { display-mode: "form" }
#@markdown ### URL file
direct_url = "" #@param {type:"string"}

#@markdown ---
#@markdown ### (Opsional) Nama file override — kosongkan untuk auto
direct_filename = "" #@param {type:"string"}


if direct_url.strip():
    print(f'📥 Download dari URL...')
    try:
        resp = requests.get(direct_url.strip(), stream=True, timeout=300, allow_redirects=True)
        resp.raise_for_status()
        if direct_filename.strip():
            fname = direct_filename.strip()
        else:
            cd = resp.headers.get('Content-Disposition', '')
            m = re.search(r'filename[*]?=["\']?([^"\';\n]+)', cd)
            if m:
                fname = urllib.parse.unquote(m.group(1).strip())
            else:
                parsed = urllib.parse.urlparse(direct_url.strip())
                fname = Path(parsed.path).name or hashlib.md5(direct_url.encode()).hexdigest()[:12]
        dest = UPLOAD_DIR / fname
        total = 0
        with open(dest, 'wb') as f:
            for chunk in resp.iter_content(chunk_size=1024*1024):
                f.write(chunk)
                total += len(chunk)
        print(f'  ✅ {fname}  ({total:,} bytes / {total/1024/1024:.1f} MB)')
    except Exception as e:
        print(f'  ❌ Error: {e}')
else:
    print('⏭️  Isi direct_url di form sebelah kanan, lalu jalankan ulang.')

## 5 — Upload Manual

In [ ]:
#@title Upload File dari PC { display-mode: "form" }
#@markdown Jalankan cell ini untuk upload file langsung dari komputer.
try:
    from google.colab import files
    print('📤 Upload file (video/audio/subtitle):')
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = UPLOAD_DIR / name
        with open(dest, 'wb') as f:
            f.write(data)
        print(f'  ✅ {name}  ({len(data):,} bytes)')
except ImportError:
    print('⚠️  Bukan di Colab — skip upload.')

## 6 — Lihat File & Register Track

In [ ]:
#@title Lihat Semua File { display-mode: "form" }
#@markdown Klik **Run** untuk melihat file yang sudah terkumpul di `/content/uploads/`
files_list = sorted(UPLOAD_DIR.iterdir())
if files_list:
    print(f'📂 {len(files_list)} file di /content/uploads/:\n')
    for f in files_list:
        size = f.stat().st_size
        ft = detect_type(f)
        print(f'  [{ft:<9}] {f.name:<45} {size:>12,} bytes  ({size/1024/1024:.1f} MB)')
else:
    print('📂 Belum ada file. Jalankan cell download/upload di atas dulu.')

In [ ]:
#@title Register Semua Track { display-mode: "form" }
#@markdown Jalankan untuk scan semua file dan register sebagai track.
TRACK_ID_COUNTER = 0

def probe_file(filepath):
    rj = subprocess.run(
        ['mkvmerge', '-J', str(filepath)],
        capture_output=True, text=True, timeout=30
    )
    tracks_json = []
    if rj.returncode == 0 and rj.stdout.strip():
        try:
            dj = json.loads(rj.stdout)
            for tr in dj.get('tracks', []):
                pr = tr.get('properties', {}) or {}
                tracks_json.append({'mkvmerge_id': tr.get('id', 0), 'codec': str(tr.get('codec', '')), 'type': str(tr.get('type', '')).lower(), 'language': str(pr.get('language', 'und')).lower(), 'track_name': str(pr.get('track_name', '') or ''), 'default_track': 'yes' if pr.get('default_track', False) else 'no'})
        except Exception:
            pass
    result = subprocess.run(
        ['mkvmerge', '--identify-verbose', str(filepath)],
        capture_output=True, text=True, timeout=30
    )
    return {'tracks_json': tracks_json, 'stdout': result.stdout, 'stderr': result.stderr, 'returncode': result.returncode}


def parse_tracks_from_probe(probe):
    tracks = []
    # Cek stdout DAN stderr (mkvmerge kadang output ke stderr)
    for text in [probe['stdout'], probe['stderr']]:
        for line in text.splitlines():
            m = re.match(r'\s*Track ID (\d+): (.+?)\s+\((.+?)\)', line)
            if m:
                tid = int(m.group(1))
                # Hindari duplikat
                if not any(t['mkvmerge_id'] == tid for t in tracks):
                    tracks.append({'mkvmerge_id': tid, 'codec': m.group(2).strip(), 'type': m.group(3).strip().lower()})
    return tracks


def register_file(filepath):
    global TRACK_ID_COUNTER
    entries = []
    file_type = detect_type(filepath)
    probe = probe_file(filepath)
    detected = parse_tracks_from_probe(probe)
    if not detected:
        detected = [{'mkvmerge_id': 0, 'codec': file_type, 'type': file_type}]
    for d in detected:
        t_raw = d['type']
        if t_raw == 'subtitles': t_raw = 'subtitle'
        track_type = t_raw if t_raw in ('video','audio','subtitle') else file_type
        entry = {
            'local_id': TRACK_ID_COUNTER,
            'source_file': str(filepath),
            'source_name': filepath.name,
            'mkvmerge_track_id': d['mkvmerge_id'],
            'codec': d['codec'],
            'type': track_type,
            'language': d.get('language', 'und'),
            'track_name': d.get('track_name', ''),
            'default_track': d.get('default_track', 'no'),
            'forced': 'no',
            'hearing_impaired': 'no',
            'visual_impaired': 'no',
            'commentary': 'no',
            'original': 'no',
            'delay': 0,
            'copy': 'no',
            'enabled': True,
        }
        TRACK_ID_COUNTER += 1
        entries.append(entry)
    return entries


all_tracks = []
for fp in sorted(UPLOAD_DIR.iterdir()):
    if fp.is_file():
        print(f'🔍 {fp.name}')
        entries = register_file(fp)
        for e in entries:
            print(f'   → Track {e["local_id"]}: {e["type"]} — {e["codec"]}')
        all_tracks.extend(entries)

print(f'\n📋 Total {len(all_tracks)} track terdaftar.')

## 7 — Lihat & Edit Track

In [ ]:
#@title Lihat Semua Track { display-mode: "form" }
def print_tracks():
    if not all_tracks:
        print('(kosong)')
        return
    print(f'{"ID":<4} {"Type":<10} {"Codec":<20} {"Source":<30} {"Lang":<5} {"Name":<20} {"Default":<8} {"Forced":<7} {"Delay":<10} {"En":<4}')
    print('─' * 130)
    for t in all_tracks:
        en = '✅' if t['enabled'] else '❌'
        print(f'{t["local_id"]:<4} {t["type"]:<10} {t["codec"]:<20} {t["source_name"]:<30} {t["language"]:<5} {t["track_name"]:<20} {t["default_track"]:<8} {t["forced"]:<7} {t["delay"]:>8}ms {en}')
print_tracks()

In [ ]:
#@title Edit Track { display-mode: "form" }
#@markdown ### Pilih track yang mau diedit
track_id = 0 #@param {type:"integer"}

#@markdown ### Bahasa (ISO 639-1)
language = "und" #@param ["und", "id", "en", "ja", "ko", "zh", "ms", "ar", "de", "fr", "es", "pt", "ru", "it", "th", "vi", "hi", "tr", "pl", "nl"]

#@markdown ### Nama Track
track_name = "" #@param {type:"string"}

#@markdown ### Default Track
default_track = "no" #@param ["yes", "no"]

#@markdown ### Forced
forced = "no" #@param ["yes", "no"]

#@markdown ### Delay (ms, positif=tunda, negatif=maju)
delay = 0 #@param {type:"integer"}

#@markdown ### Flags lainnya
hearing_impaired = "no" #@param ["yes", "no"]
visual_impaired = "no" #@param ["yes", "no"]
commentary = "no" #@param ["yes", "no"]
original = "no" #@param ["yes", "no"]

#@markdown ### Aktifkan track ini?
enabled = True #@param {type:"boolean"}


found = False
for t in all_tracks:
    if t['local_id'] == track_id:
        t['language'] = language
        if track_name.strip(): t['track_name'] = track_name.strip()
        t['default_track'] = default_track
        t['forced'] = forced
        t['delay'] = delay
        t['hearing_impaired'] = hearing_impaired
        t['visual_impaired'] = visual_impaired
        t['commentary'] = commentary
        t['original'] = original
        t['enabled'] = enabled
        found = True
        break

if found:
    print(f'✅ Track {track_id} updated.')
    print_tracks()
else:
    print(f'❌ Track {track_id} tidak ditemukan.')

In [ ]:
#@title Batch Edit — Terapkan ke Banyak Track { display-mode: "form" }
#@markdown ### Pilih track yang mau diedit
target_type = "Semua" #@param ["Semua", "Video", "Audio", "Subtitle"]
target_ids = "" #@param {type:"string"}

#@markdown ---
#@markdown ### Yang mau diubah (kosongkan jika tidak diubah)
batch_language = "" #@param ["", "id", "en", "ja", "ko", "zh", "ms", "ar", "de", "fr", "es", "pt", "ru", "it", "th", "vi", "hi", "tr", "pl", "nl"]
batch_track_name = "" #@param {type:"string"}
batch_default = "" #@param ["", "yes", "no"]
batch_forced = "" #@param ["", "yes", "no"]
batch_delay = 0 #@param {type:"integer"}

#@markdown ---
#@markdown ### Flags (isi `yes` atau `kosongkan`)
batch_hearing_impaired = "" #@param ["", "yes", "no"]
batch_visual_impaired = "" #@param ["", "yes", "no"]
batch_commentary = "" #@param ["", "yes", "no"]
batch_original = "" #@param ["", "yes", "no"]

#@markdown ---
#@markdown ### ✅ Centang untuk apply
apply_batch = False #@param {type:"boolean"}


if not apply_batch:
    print('ℹ️  Centang apply_batch dulu, lalu jalankan ulang.')
else:
    # Parse target IDs
    selected_ids = set()
    if target_ids.strip():
        for part in target_ids.split(','):
            part = part.strip()
            if '-' in part:
                start, end = part.split('-', 1)
                selected_ids.update(range(int(start), int(end) + 1))
            elif part.isdigit():
                selected_ids.add(int(part))

    # Map type
    type_map = {'Semua': None, 'Video': 'video', 'Audio': 'audio', 'Subtitle': 'subtitle'}
    target_t = type_map[target_type]

    count = 0
    for t in all_tracks:
        # Filter by type
        if target_t and t['type'] != target_t:
            continue
        # Filter by IDs (if specified)
        if selected_ids and t['local_id'] not in selected_ids:
            continue

        # Apply changes
        if batch_language:          t['language'] = batch_language
        if batch_track_name.strip(): t['track_name'] = batch_track_name.strip()
        if batch_default:            t['default_track'] = batch_default
        if batch_forced:             t['forced'] = batch_forced
        if batch_delay != 0:         t['delay'] = batch_delay
        if batch_hearing_impaired:   t['hearing_impaired'] = batch_hearing_impaired
        if batch_visual_impaired:    t['visual_impaired'] = batch_visual_impaired
        if batch_commentary:         t['commentary'] = batch_commentary
        if batch_original:           t['original'] = batch_original
        count += 1

    print(f'✅ Batch edit: {count} track diupdate.\n')
    print_tracks()

In [ ]:
#@title Atur Default Track { display-mode: "form" }
#@markdown ### Pilih track yang mau dijadikan default
#@markdown Jalankan cell "Lihat Semua Track" dulu untuk melihat ID.
set_default_id = -1 #@param {type:"integer"}

#@markdown ### Atau: reset semua default ke "No" dulu
clear_all_defaults = False #@param {type:"boolean"}


if clear_all_defaults:
    for t in all_tracks:
        t['default_track'] = 'no'
    print('🔄 Semua default track direset ke "no".\n')

if set_default_id >= 0:
    found = False
    for t in all_tracks:
        if t['local_id'] == set_default_id:
            target_type = t['type']
            # Clear default lain yang se-tipe
            cleared = 0
            for other in all_tracks:
                if other['type'] == target_type and other['default_track'] == 'yes':
                    other['default_track'] = 'no'
                    cleared += 1
            t['default_track'] = 'yes'
            print(f'✅ Track {set_default_id} ({t["source_name"]}) dijadikan default {target_type}.')
            if cleared:
                print(f'   🔄 {cleared} track {target_type} lain direset ke "no".')
            found = True
            break
    if not found:
        print(f'❌ Track {set_default_id} tidak ditemukan.')

if set_default_id < 0 and not clear_all_defaults:
    print('ℹ️  Isi set_default_id atau centang clear_all_defaults, lalu jalankan ulang.')

print()
print_tracks()

In [ ]:
#@title Tambah / Hapus Track { display-mode: "form" }
#@markdown ### Duplikat track
dup_track_id = -1 #@param {type:"integer"}

#@markdown ### Hapus track
del_track_id = -1 #@param {type:"integer"}

if dup_track_id >= 0:
    for t in all_tracks:
        if t['local_id'] == dup_track_id:
            new_t = dict(t)
            new_t['local_id'] = TRACK_ID_COUNTER
            TRACK_ID_COUNTER += 1
            all_tracks.append(new_t)
            print(f'✅ Track {dup_track_id} diduplikasi → ID baru {new_t["local_id"]}')
            break
    else:
        print(f'❌ Track {dup_track_id} tidak ditemukan.')

if del_track_id >= 0:
    before = len(all_tracks)
    all_tracks = [t for t in all_tracks if t['local_id'] != del_track_id]
    if len(all_tracks) < before:
        print(f'🗑️  Track {del_track_id} dihapus.')
    else:
        print(f'❌ Track {del_track_id} tidak ditemukan.')

if dup_track_id < 0 and del_track_id < 0:
    print('ℹ️  Isi dup_track_id atau del_track_id di form, lalu jalankan ulang.')

print()
print_tracks()

## 8 — Mux

In [ ]:
#@title Konfigurasi Output { display-mode: "form" }
#@markdown ### Nama file output (kosongkan = otomatis dari nama video)
output_filename = "" #@param {type:"string"}

# Auto-detect dari file video pertama
if not output_filename.strip():
    video_tracks = [t for t in all_tracks if t['type'] == 'video']
    if video_tracks:
        video_stem = Path(video_tracks[0]['source_name']).stem
        output_filename = video_stem + '.mkv'
    else:
        output_filename = 'output.mkv'

OUTPUT_PATH = OUTPUT_DIR / output_filename
print(f'📁 Output: {OUTPUT_PATH}')

In [ ]:
#@title Mux Sekarang { display-mode: "form" }
#@markdown ### Auto-fix default track? (recommended)
#@markdown Satu tipe = satu default. Jika ada lebih dari 1, yang pertama dipertahankan.
auto_fix_default = True #@param {type:"boolean"}

def enforce_single_default_per_type():
    """Pastikan per tipe (video/audio/subtitle) cuma ada 1 default track."""
    fixed = 0
    for track_type in ['video', 'audio', 'subtitle']:
        defaults = [t for t in all_tracks if t['type'] == track_type and t['default_track'] == 'yes']
        if len(defaults) > 1:
            for t in defaults[1:]:
                t['default_track'] = 'no'
                fixed += 1
        elif len(defaults) == 0:
            # Belum ada default → set yang pertama
            first = next((t for t in all_tracks if t['type'] == track_type), None)
            if first:
                first['default_track'] = 'yes'
                fixed += 1
    return fixed

def build_mux_command():
    by_file = {}
    for t in all_tracks:
        if not t['enabled']:
            continue
        by_file.setdefault(t['source_file'], []).append(t)
    cmd = ['mkvmerge', '-o', str(OUTPUT_PATH)]
    for filepath, tracks in by_file.items():
        cmd.extend(['--no-chapters', '--no-global-tags'])
        for t in tracks:
            tid = str(t['mkvmerge_track_id'])
            if t['track_name']:
                cmd.extend(['--track-name', f'{tid}:{t["track_name"]}'])
            if t['language'] and t['language'] != 'und':
                cmd.extend(['--language', f'{tid}:{t["language"]}'])
            if t['default_track'] != 'auto':
                cmd.extend(['--default-track', f'{tid}:{t["default_track"]}'])
            if t['forced'] == 'yes':
                cmd.extend(['--forced-track', f'{tid}:yes'])
            if t['hearing_impaired'] == 'yes':
                cmd.extend(['--hearing-impaired-flag', f'{tid}:yes'])
            if t['visual_impaired'] == 'yes':
                cmd.extend(['--visual-impaired-flag', f'{tid}:yes'])
            if t['commentary'] == 'yes':
                cmd.extend(['--commentary-flag', f'{tid}:yes'])
            if t['original'] == 'yes':
                cmd.extend(['--original-flag', f'{tid}:yes'])
            if t['delay'] != 0:
                cmd.extend(['--sync', f'{tid}:{t["delay"]:+d}'])
        cmd.append(filepath)
    return cmd

if auto_fix_default:
    fixed = enforce_single_default_per_type()
    if fixed:
        print(f'🔧 Auto-fix: {fixed} default track direset (hanya 1 per tipe)\n')

cmd = build_mux_command()
print('🚀 Mulai muxing...\n')
result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

# Cek apakah output file berhasil dibuat (warning ≠ error)
mux_success = OUTPUT_PATH.exists() and OUTPUT_PATH.stat().st_size > 0

if mux_success:
    file_size = OUTPUT_PATH.stat().st_size
    print(f'✅ Muxing berhasil!')
    print(f'   📄 {OUTPUT_PATH.name}  ({file_size:,} bytes / {file_size/1024/1024:.1f} MB)')
    # Tampilkan warning jika ada (bukan error)
    warnings = [l for l in result.stdout.splitlines() if 'Warning' in l]
    if warnings:
        print(f'\n⚠️  {len(warnings)} warning(s):')
        for w in warnings[:3]:
            print(f'   {w[:100]}')
else:
    print(f'❌ Muxing gagal!')
    print('STDOUT:', result.stdout[-500:] if result.stdout else '')
    print('STDERR:', result.stderr[-500:] if result.stderr else '')

## 8B — MediaInfo (cek hasil)

In [ ]:
#@title Cek MediaInfo { display-mode: "form" }
#@markdown ### Path file (otomatis = hasil muxing terakhir)
mediainfo_path = "" #@param {type:"string"}

#@markdown ### Format output
mediainfo_format = "Text" #@param ["Text", "JSON"]


def get_mediainfo(filepath, fmt='text'):
    """Jalankan mediainfo dan return output."""
    cmd = ['mediainfo']
    if fmt == 'json':
        cmd.append('--Output=JSON')
    cmd.append(str(filepath))
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=30)
    return result.stdout


def parse_mediainfo_tracks(info_text):
    """Parse mediainfo text output jadi list track info."""
    tracks = []
    current_type = None
    current_data = {}
    section_headers = {'General', 'Video', 'Audio', 'Text', 'Menu', 'Image'}
    for line in info_text.splitlines():
        stripped = line.strip()
        if not stripped:
            continue
        first_word = stripped.split()[0] if stripped.split() else ''
        # Handle 'Audio #1', 'Text #2' etc.
        is_header = first_word in section_headers and (':' not in stripped or stripped.startswith(first_word))
        if is_header:
            if current_type and current_data:
                tracks.append(current_data)
            current_type = stripped
            current_data = {'type': stripped}
            continue
        if ':' in stripped and current_type:
            key, val = stripped.split(':', 1)
            key, val = key.strip(), val.strip()
            if key and val:
                current_data[key] = val
    if current_type and current_data:
        tracks.append(current_data)
    return tracks


# Resolve path
if mediainfo_path.strip():
    mi_path = Path(mediainfo_path.strip())
else:
    mi_path = OUTPUT_PATH

if not mi_path.exists():
    print(f'❌ File tidak ditemukan: {mi_path}')
else:
    print(f'📋 MediaInfo: {mi_path.name}\n')
    fmt = 'json' if mediainfo_format == 'JSON' else 'text'
    info = get_mediainfo(mi_path, fmt)

    if fmt == 'json':
        data = json.loads(info)
        general = data.get('media', {}).get('track', [{}])[0]
        print(f'Format: {general.get("Format", "?")}')
        print(f'Size: {general.get("FileSize", "?")} bytes')
        print(f'Duration: {general.get("Duration", "?")}s')
        print(f'Bitrate: {general.get("OverallBitRate", "?")} bps')
        print()
        for t in data.get('media', {}).get('track', [])[1:]:
            ttype = t.get('Track type', '?')
            codec = t.get('Format', t.get('CodecID', '?'))
            lang = t.get('Language', '-')
            name = t.get('Title', '-')
            default = t.get('Default', '-')
            forced = t.get('Forced', '-')
            print(f'  [{ttype:<11}] {codec:<25} Lang:{lang:<6} Name:{name:<20} Default:{default}  Forced:{forced}')
    else:
        tracks = parse_mediainfo_tracks(info)
        for t in tracks:
            ttype = t.get('Type', '?')
            codec = t.get('Format', t.get('CodecID', '?'))
            lang = t.get('Language', '-')
            name = t.get('Title', '-')
            default = t.get('Default', '-')
            forced = t.get('Forced', '-')
            print(f'  [{ttype:<11}] {codec:<25} Lang:{lang:<6} Name:{name:<20} Default:{default}  Forced:{forced}')

## 9 — Upload Hasil

In [ ]:
#@title Upload ke Gofile (Guest) { display-mode: "form" }
#@markdown ### File yang mau di-upload (path lengkap)
#@markdown Kosongkan untuk upload hasil muxing terakhir.
upload_file_path = "" #@param {type:"string"}


def gofile_get_server():
    resp = requests.get('https://api.gofile.io/servers', timeout=15)
    data = resp.json()
    if data.get('status') == 'ok':
        return data['data']['servers'][0]['name']
    return 'store1'


def gofile_upload(filepath):
    if not filepath.exists():
        print(f'  ❌ File tidak ditemukan: {filepath}')
        return None
    server = gofile_get_server()
    upload_url = f'https://{server}.gofile.io/uploadfile'
    size_mb = filepath.stat().st_size / 1024 / 1024
    print(f'  📤 Upload ke {server}.gofile.io ... ({filepath.name}, {size_mb:.1f} MB)')
    try:
        with open(filepath, 'rb') as f:
            resp = requests.post(upload_url, files={'file': (filepath.name, f)}, timeout=600)
        result = resp.json()
        if result.get('status') == 'ok':
            d = result['data']
            print(f'  ✅ Upload berhasil!')
            print(f'     Download: {d["downloadPage"]}')
            print(f'     Code: {d["code"]}')
            return d
        else:
            print(f'  ❌ Upload gagal: {json.dumps(result, indent=2)}')
            return None
    except Exception as e:
        print(f'  ❌ Error: {e}')
        return None


fp = Path(upload_file_path.strip()) if upload_file_path.strip() else OUTPUT_PATH
print(f'📤 Upload ke Gofile:\n')
result = gofile_upload(fp)
if result:
    print(f'\n📋 Link: {result["downloadPage"]}')

In [ ]:
#@title Upload ke Google Drive { display-mode: "form" }
#@markdown ### Folder tujuan di MyDrive
gdrive_upload_folder = "HaruColab" #@param {type:"string"}

#@markdown ### File yang mau di-upload (path lengkap, kosongkan untuk hasil muxing)
gdrive_upload_file = "" #@param {type:"string"}


def ensure_drive_mounted():
    if Path('/content/drive').exists() and any(Path('/content/drive').iterdir()):
        return True
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        return True
    except Exception as e:
        print(f'❌ Gagal mount Drive: {e}')
        return False


fp = Path(gdrive_upload_file.strip()) if gdrive_upload_file.strip() else OUTPUT_PATH
if ensure_drive_mounted():
    dest_dir = Path(f'/content/drive/MyDrive/{gdrive_upload_folder.strip()}')
    dest_dir.mkdir(parents=True, exist_ok=True)
    if fp.exists():
        shutil.copy2(fp, dest_dir / fp.name)
        print(f'✅ {fp.name}  →  /content/drive/MyDrive/{gdrive_upload_folder.strip()}/')
    else:
        print(f'❌ File tidak ditemukan: {fp}')
else:
    print('❌ Tidak bisa mount Google Drive.')

## 10 — Download Hasil ke PC

In [ ]:
#@title Download ke PC { display-mode: "form" }
try:
    from google.colab import files
    if OUTPUT_PATH.exists():
        files.download(str(OUTPUT_PATH))
    else:
        print('❌ File output tidak ditemukan.')
except ImportError:
    print(f'📂 File ada di: {OUTPUT_PATH}')